# Mission 05 — Dual-Path GraphRAG, Hybrid Search & NotebookLM Grounding

**USSE Noosphere — Week 05.** Integrated four-station exercise.

This notebook imports the **actual source files in this folder**. It demonstrates a small executable HNSW+BM25/RRF pipeline and safe-context scoring. Google NotebookLM tests require manual use of the actual three source documents and are **not claimed as complete**.


In [ ]:
import importlib.util
import json
from pathlib import Path
import numpy as np

ROOT = Path.cwd().resolve()
if not (ROOT / '01_HNSW_Vector_Index.py').is_file():
    raise FileNotFoundError('Open the notebook with its working directory set to the problem_set folder.')

def load_module(filename, name):
    spec = importlib.util.spec_from_file_location(name, ROOT / filename)
    if spec is None or spec.loader is None:
        raise ImportError(filename)
    obj = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(obj)
    return obj

hnsw = load_module('01_HNSW_Vector_Index.py', 'ps05_hnsw')
hybrid = load_module('02_BM25_and_RRF_Combiner.py', 'ps05_hybrid')
notebooklm = load_module('03_NotebookLM_RAG_Audit.py', 'ps05_notebooklm')
security = load_module('04_RAG_Provenance_and_Safety.py', 'ps05_security')
print('Loaded all four Mission 5 modules.')

## Station 1 — HNSW graph and cosine proximity (25 pts)

Unit-normalized cosine vectors, layer law $\ell=\lfloor-\ln(U) / \ln(M)\rfloor$, multilevel greedy traversal and beam search. The shipped `run_output.txt` includes a previously recorded 25,000 × 768 benchmark; executing the optional block below produces fresh timings for your machine.

In [ ]:
X = np.asarray([[1, 0], [0, 1], [0.9, 0.1], [-1, 0]], dtype=np.float32)
index = hnsw.HNSWIndex(dim=2, M=4, ef_construction=16, capacity=len(X), seed=7)
index.add_items(X)
q = np.asarray([1, 0], dtype=np.float32)
hnsw_ids, hnsw_sim = index.search(q, k=3, ef=20)
flat_ids, flat_sim = hnsw.FlatIndex(X).search(q, k=3)
print('HNSW IDs/similarities:', hnsw_ids.tolist(), hnsw_sim.tolist())
print('Flat  IDs/similarities:', flat_ids.tolist(), flat_sim.tolist())
assert int(hnsw_ids[0]) == int(flat_ids[0]) == 0
print('Nodes per graph level:', index.layer_sizes())

In [ ]:
# Fast, independently runnable smoke benchmark using actual Part 1 API.
small = hnsw.run_benchmark(N=200, dim=32, n_queries=4, k=5, M=4, ef_construction=16, make_plot=False)
print('Flat ms/query:', small['flat_ms'])
print('HNSW ef=50 ms/query:', next(row['ms'] for row in small['rows'] if row['ef'] == 50))

# Part 1 has already produced run_output.txt & hnsw_benchmark.png at N=25,000, d=768.
# Optional fresh run (longer): set True on your machine.
RUN_FULL_25K_BENCHMARK = False
if RUN_FULL_25K_BENCHMARK:
    full = hnsw.run_benchmark(N=25_000, dim=768, n_queries=200, k=10, make_plot=False)
    summary = {'N': full['N'], 'd': 768, 'build_s': full['build_s'],
               'flat_ms': full['flat_ms'], 'rows': full['rows'], 'sizes': full['sizes']}
    (ROOT / 'results').mkdir(exist_ok=True)
    (ROOT / 'results' / 'hnsw_25k_benchmark.json').write_text(json.dumps(summary, indent=2))
    print('Saved fresh 25k benchmark summary')
else:
    print('Using shipped 25k benchmark evidence in run_output.txt; no fresh 25k run requested.')

## Station 2 — HNSW dense + BM25 sparse + RRF (25 pts)

Exact identifiers are kept intact by `tokenize`, and BM25 uses $k_1=1.5$, $b=0.75$. The vector path below uses **the actual HNSWIndex from Part 1**, not just Part 2's flat dense baseline. The vectors come from Part 2's deterministic educational encoder, not a pretrained embedding model. RRF has smoothing constant $k=60$.

In [ ]:
corpus = hybrid.CORPUS
lexical = hybrid.BM25(corpus)
embeddings = np.vstack([hybrid.embed(text) for text in corpus]).astype(np.float32)
dense_hnsw = hnsw.HNSWIndex(dim=embeddings.shape[1], M=4, ef_construction=32,
                            capacity=len(corpus), seed=3)
dense_hnsw.add_items(embeddings)


def retrieve_fused(question, depth=10):
    dense_ids, similarities = dense_hnsw.search(hybrid.embed(question).astype(np.float32),
                                                k=min(depth, len(corpus)), ef=40)
    sparse_results = lexical.search(question, top_k=depth)
    dense_rank = [int(i) for i in dense_ids]
    sparse_rank = [int(i) for i, _ in sparse_results]
    fused = hybrid.reciprocal_rank_fusion([dense_rank, sparse_rank], k=60)
    return {'dense_hnsw': dense_rank, 'sparse_bm25': sparse_rank, 'fused_rrf': fused}

conceptual = retrieve_fused('What makes the ship accelerate through space?')
exact = retrieve_fused('QX-7731-B')
print('Conceptual:', conceptual)
print('Exact serial code:', exact)
assert hybrid.tokenize('QX-7731-B') == ['qx-7731-b']
assert lexical.search('QX-7731-B', top_k=1)[0][0] == 1
assert 1 in [doc for doc, _ in exact['fused_rrf'][:3]]
assert 0 in [doc for doc, _ in conceptual['fused_rrf'][:3]]
assert len(exact['dense_hnsw']) > 0 and len(exact['sparse_bm25']) > 0
print('PASS: combined HNSW + BM25 + RRF retrieves concept and exact identifier')

## Station 3 — NotebookLM grounding/citations/negative probes/audio (25 pts)

The file below is a **blank audit template**. Upload original Lewis (2020), Edge (2024), and the actual course-provided Noosphere archive to Google NotebookLM. Ask the five cross-document and three absent-fact questions. Click all citation badges and log whether the exact highlighted span supports each claim. Record a real, approximately 120-second two-speaker Audio Overview transcription and 1–5 scores. Do not invent results or claim success from unfilled data.

In [ ]:
observations_path = ROOT / 'data' / 'notebooklm_observations.json'
if observations_path.exists():
    observations = json.loads(observations_path.read_text(encoding='utf-8'))
else:
    observations_path.parent.mkdir(exist_ok=True)
    observations = notebooklm.create_template()
    observations_path.write_text(json.dumps(observations, indent=2) + '\n', encoding='utf-8')
    print('Created unfilled NotebookLM audit template:', observations_path)

print('Documents:', [(d['id'], d['uploaded']) for d in observations['documents']])
print('5 cross-document prompts:', len(observations['cross_document']))
print('3 negative prompts:', len(observations['negative_constraints']))
metrics = notebooklm.evaluate(observations)
print('NotebookLM evaluation:', json.dumps(metrics, indent=2))
assert metrics['status'] in ('needs_human_observations', 'complete')

## Station 4 — Prompt injection screening, RAG Triad and provenance (15 pts)

Retrieved passages are untrusted. Part 4 discards suspicious chunks and writes JSONL provenance. Its context relevance, faithfulness, and answer relevance scores are **lexical/hashed-vector proxies**; they are not a trained NLI proof of entailment or a production security boundary.

In [ ]:
record = security.audit_turn(
    'What is the coolant pump part identifier?',
    security.DEMO_CHUNKS,
    log_path=ROOT / 'results' / 'provenance.jsonl',
)
print(json.dumps({k: record[k] for k in (
    'query', 'retrieved_chunk_ids', 'accepted_chunk_ids', 'blocked_chunk_ids',
    'final_grounded_answer', 'rag_triad')}, indent=2))
assert 'attack-02' in record['blocked_chunk_ids']
assert 'attack-02' not in record['accepted_chunk_ids']
assert all(0 <= record['rag_triad'][metric] <= 1 for metric in (
    'context_relevance', 'faithfulness', 'answer_relevance'))
print('PASS: malicious context blocked, triad scores present, provenance log written')

## Final checkpoint and four sign-offs (10 pts)

- [ ] **Systems Architect:** inspect already recorded 25k × 768 HNSW benchmark in `run_output.txt` and `hnsw_benchmark.png`; sign (name/date/commit).
- [ ] **Protocol Officer:** confirm conceptual + exact code retrieval and RRF(k=60); sign (name/date/commit).
- [ ] **Data Specialist:** upload all 3 original sources in NotebookLM, verify all citation highlights, log 5 cross-source and 3 negative responses, transcribe and rate a real 2-minute Audio Overview; sign (name/date/commit).
- [ ] **Security Sentinel:** validate injection filter + triad + JSON provenance; sign (name/date/commit).

**Known incomplete step:** The empirical NotebookLM audit cannot be completed from code alone. Review `notebooklm_audit_report.md` and populate `data/notebooklm_observations.json` with your actual results. Never present this as observed data before running it.